In [1]:
import kagglehub
import pandas as pd
import os

# downloads to a local cache directory, returns the path
path = kagglehub.dataset_download("mashlyn/online-retail-II-uci")
print(path)
print(os.listdir(path))

/Users/ryanmccurry/Desktop/retail-churn-prediction/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/ryanmccurry/Desktop/retail-churn-prediction/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/ryanmccurry/.cache/kagglehub/datasets/mashlyn/online-retail-II-uci/versions/3
['online_retail_II.csv']


In [2]:
df = pd.read_csv(os.path.join(path, "online_retail_II.csv"))

print(f'Shape: {df.shape}')
df.head()

Shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [4]:
cancellations = df[df['Invoice'].astype(str).str.startswith('C')]
print(f'Number of cancellation rows: {len(cancellations):,}')
print(f'Pencent of total: {len(cancellations) / len(df) * 100:.2f}%')
cancellations.head()

Number of cancellation rows: 19,494
Pencent of total: 1.83%


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


In [5]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'Start date:         {df["InvoiceDate"].min().date()}')
print(f'End date:           {df["InvoiceDate"].max().date()}')
print(f'Date range span:    {(df["InvoiceDate"].max() - df["InvoiceDate"].min()).days} days')

Start date:         2009-12-01
End date:           2011-12-09
Date range span:    738 days


In [6]:
n_customers = df['Customer ID'].nunique()
print(f'Number of unique customers: {n_customers}')

Number of unique customers: 5942


In [7]:
invoices_per_customer = df.groupby('Customer ID')['Invoice'].nunique()
print(f'Average invoices per customer:  {invoices_per_customer.mean():.2f}')
print(f'Median invoices per customer:   {invoices_per_customer.median():.2f}')

Average invoices per customer:  7.55
Median invoices per customer:   4.00


In [8]:
df_clean = df.dropna(subset=['Customer ID']).copy()
print(f'Rows before: {len(df):,}')
print(f'Rows after: {len(df_clean):,}')
print(f'Rows dropped: {len(df) - len(df_clean):,}')

Rows before: 1,067,371
Rows after: 824,364
Rows dropped: 243,007


In [9]:
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)
df_clean['Customer ID'].head()

0    13085
1    13085
2    13085
3    13085
4    13085
Name: Customer ID, dtype: int64

**Cancellation Handling**: Rather than discarding cancelled orders, we split them into a separate dataframe (`cancellations_clean`) to preserve return behavior as a potential churn signal. Purchases and cancellations will be used together during feature engineering (e.g., a customer's return rate may correlate with future churn)

In [10]:
# split cancellations out from actual purchases
cancellations_clean = df_clean[df_clean['Invoice'].astype(str).str.startswith('C')].copy()
purchases_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')].copy()

print(f'Purchases: {len(purchases_clean):,} rows')
print(f'Cancellations: {len(cancellations_clean):,} rows')
print(f'Total (should match df_clean): {len(purchases_clean) + len(cancellations_clean):,}')

Purchases: 805,620 rows
Cancellations: 18,744 rows
Total (should match df_clean): 824,364


In [11]:
missing_description = purchases_clean['Description'].isnull().sum()
print(f'Missing descriptions in purchases_clean: {missing_description}')

Missing descriptions in purchases_clean: 0


In [12]:
last_date = purchases_clean['InvoiceDate'].max()
reference_date = last_date - pd.Timedelta(days=180)
print(f'Last date in date: {last_date}')
print(f'Reference date: {reference_date}')

Last date in date: 2011-12-09 12:50:00
Reference date: 2011-06-12 12:50:00


In [13]:
# split into pre-reference (features) and post-reference (outcome) windows
pre_reference = purchases_clean[purchases_clean['InvoiceDate'] <= reference_date]
post_reference = purchases_clean[purchases_clean['InvoiceDate'] > reference_date]

# customers active before the reference date
active_customers = pre_reference['Customer ID'].unique()
print(f'Active customers as of reference date: {len(active_customers):,}')

# customers who purchased again in the 90-day window after
returning_customers = post_reference['Customer ID'].unique()
print(f'Customers with purchases within the outcome window: {len(returning_customers):,}')

# build the label
churn_labels = pd.DataFrame({'Customer ID': active_customers})
churn_labels['churned'] = (~churn_labels['Customer ID'].isin(returning_customers)).astype(int)

print(f'\nChurn rate: {churn_labels["churned"].mean():.2%}')
churn_labels['churned'].value_counts()

Active customers as of reference date: 4,979
Customers with purchases within the outcome window: 3,479

Churn rate: 48.24%


churned
0    2577
1    2402
Name: count, dtype: int64

Tested both 90-day and 180-day churn windows. Given the median purchase cycle of ~180 days observed in EDA, a 90-day window over-flags naturally infrequent buyers as churned; a 180-day window better reflects this dataset's actual purchase rhythm.

In [14]:
recency = pre_reference.groupby('Customer ID')['InvoiceDate'].max().reset_index()
recency.columns = ['Customer ID', 'last_purchase_date']
recency['recency_days'] = (reference_date - recency['last_purchase_date']).dt.days

print(recency['recency_days'].describe())
recency.head()

count    4979.000000
mean      172.472183
std       146.117651
min         0.000000
25%        37.500000
50%       146.000000
75%       251.500000
max       558.000000
Name: recency_days, dtype: float64


,Customer ID,last_purchase_date,recency_days
0,12346,2011-01-18 10:01:00,145
1,12347,2011-06-09 13:01:00,2
2,12348,2011-04-05 10:47:00,68
3,12349,2010-10-28 08:23:00,227
4,12350,2011-02-02 16:01:00,129


In [15]:
frequency = pre_reference.groupby('Customer ID')['Invoice'].nunique().reset_index()
frequency.columns = ['Customer ID', 'frequency']

print(frequency['frequency'].describe())
frequency.head()

count    4979.000000
mean        5.300462
std        10.268067
min         1.000000
25%         1.000000
50%         3.000000
75%         6.000000
max       263.000000
Name: frequency, dtype: float64


,Customer ID,frequency
0,12346,12
1,12347,5
2,12348,4
3,12349,3
4,12350,1


In [16]:
pre_reference = pre_reference.copy()
pre_reference['line_total'] = pre_reference['Quantity'] * pre_reference['Price']

monetary = pre_reference.groupby('Customer ID')['line_total'].sum().reset_index()
monetary.columns = ['Customer ID', 'monetary']

print(monetary['monetary'].describe())
monetary.head()

count      4979.000000
mean       2497.745880
std       11187.059398
min           0.000000
25%         319.385000
50%         781.590000
75%        2012.015000
max      441293.980000
Name: monetary, dtype: float64


,Customer ID,monetary
0,12346,77556.46
1,12347,3529.27
2,12348,1709.40
3,12349,2671.14
4,12350,334.40


In [17]:
monetary[monetary['monetary'] == 0]

,Customer ID,monetary
1427,14103,0.0
2052,14827,0.0


In [18]:
engineered_df = recency[['Customer ID', 'recency_days']].merge(
    frequency, on='Customer ID'
).merge(
    monetary, on='Customer ID'
).merge(
    churn_labels, on='Customer ID'
)

print(engineered_df.shape)
engineered_df.head()

(4979, 5)


,Customer ID,recency_days,frequency,monetary,churned
0,12346,145,12,77556.46,1
1,12347,2,5,3529.27,0
2,12348,68,4,1709.40,0
3,12349,227,3,2671.14,0
4,12350,129,1,334.40,1


In [19]:
# filter cancellations to before the reference date, same from purchases
cancellations_pre = cancellations_clean[cancellations_clean['InvoiceDate'] <= reference_date]

# count cancellations invoices per customer
returns = cancellations_pre.groupby('Customer ID')['Invoice'].nunique().reset_index()
returns.columns = ['Customer ID', 'return_count']

# merge into engineered_df - customers with no cancellations won't appear in 'returns' (missing values = 0)
engineered_df = engineered_df.merge(returns, on='Customer ID', how='left')
engineered_df['return_count'] = engineered_df['return_count'].fillna(0).astype(int)

# return rate = cancellations / (purchases + cancellations)
engineered_df['return_rate'] = (engineered_df['return_count'] / (engineered_df['frequency'] + engineered_df['return_count'])).round(4)

print(engineered_df['return_count'].describe())
print(engineered_df['return_rate'].describe())
engineered_df.head()

count    4979.000000
mean        1.182165
std         3.044460
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max        79.000000
Name: return_count, dtype: float64
count    4979.000000
mean        0.122852
std         0.173305
min         0.000000
25%         0.000000
50%         0.000000
75%         0.235300
max         0.800000
Name: return_rate, dtype: float64


,Customer ID,recency_days,frequency,monetary,churned,return_count,return_rate
0,12346,145,12,77556.46,1,5,0.2941
1,12347,2,5,3529.27,0,0,0.0000
2,12348,68,4,1709.40,0,0,0.0000
3,12349,227,3,2671.14,0,1,0.2500
4,12350,129,1,334.40,1,0,0.0000


In [20]:
os.makedirs('../data/processed', exist_ok=True)
engineered_df.to_csv('../data/processed/engineered_features.csv', index=False)
print(f'Saved {len(engineered_df)} rows to ../data/processed/engineered_features.csv')

Saved 4979 rows to ../data/processed/engineered_features.csv


## Modeling

With the feature table complete (`recency_days`, `frequency`, `monetary`, `return_rate`) and a defined churn label, we move into building a predictive model.

**Approach:**
1. Split the engineered feature table into train/test sets (80/20, stratified on `churned` to preserve the ~48% churn rate in both splits)
2. Establish a baseline with logistic regression — simple, interpretable, and a useful benchmark before trying more complex models
3. Iterate toward a stronger model (e.g., XGBoost or LightGBM) with proper cross-validation
4. Evaluate using both ML metrics (ROC-AUC, precision/recall) and a business-facing metric (revenue at risk among predicted churners), since the end goal is a model a stakeholder could actually act on

In [21]:
from sklearn.model_selection import train_test_split

feature_cols = ['recency_days', 'frequency', 'monetary', 'return_rate']
X = engineered_df[feature_cols]
y = engineered_df['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.2%}')
print(f'Test churn rate: {y_test.mean():.2%}')

Train shape: (3983, 4)
Test shape: (996, 4)
Train churn rate: 48.26%
Test churn rate: 48.19%


### Baseline Model: Logistic Regression

Before trying more complex models, we start with logistic regression as a baseline. It's simple, fast, and interpretable — a useful benchmark to compare against later, and a sanity check that the engineered features carry real predictive signal.

Since logistic regression is sensitive to feature scale, we standardize the features (mean 0, std 1) before fitting. The scaler is fit only on training data and applied to test data separately, to avoid leaking test-set information into preprocessing.

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# scale features - standardize features which are on very different scales (recency_days: 0-500, return_rate: 0-1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# fit baseline model
log_reg = LogisticRegression(random_state=42).fit(X_train_scaled, y_train)

# predictions
y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

print('Baseline Logistic Regression trained.')

Baseline Logistic Regression trained.


In [23]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

print('\nClassification Report:')
print(classification_report(y_test, y_pred))

print(f'\nROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.3f}')

Confusion Matrix:
[[374 142]
 [155 325]]

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.72      0.72       516
           1       0.70      0.68      0.69       480

    accuracy                           0.70       996
   macro avg       0.70      0.70      0.70       996
weighted avg       0.70      0.70      0.70       996


ROC-AUC Score: 0.791


### Model 2: XGBoost

Logistic regression assumes a linear relationship between features and the log-odds of churn. Tree-based models like XGBoost can capture non-linear patterns and interactions between features (e.g., "high monetary value AND high recency" might matter differently than either feature alone) — worth testing whether that improves on the baseline.

In [24]:
from xgboost import XGBClassifier

xgb = XGBClassifier(random_state=42, eval_metric='logloss').fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_pred_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_xgb))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_xgb))

print(f'\nROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_xgb):.3f}')

Confusion Matrix:
[[355 161]
 [153 327]]

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.69      0.69       516
           1       0.67      0.68      0.68       480

    accuracy                           0.68       996
   macro avg       0.68      0.68      0.68       996
weighted avg       0.68      0.68      0.68       996


ROC-AUC Score: 0.773


### Model Comparison

| Metric | Logistic Regression | XGBoost |
|---|---|---|
| ROC-AUC | 0.791 | 0.773 |
| Accuracy | 0.70 | 0.68 |
| Recall (churned) | 0.68 | 0.68 |
| Precision (churned) | 0.70 | 0.67 |

Logistic regression slightly outperforms XGBoost on this feature set. This is a reasonable outcome given the small number of features (4) and their fairly linear relationship with churn observed in EDA — XGBoost's strength in capturing non-linear interactions has less to exploit here, and with a relatively small dataset, it may be marginally overfitting to training noise. **Logistic regression is selected as the better-performing and more interpretable model for this feature set.**

### Tuning XGBoost

Before concluding logistic regression is the better model, we tune XGBoost's key hyperparameters using grid search with cross-validation, to make sure we're comparing against XGBoost's best version rather than its untuned defaults.

In [25]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 4],
    'learning_rate': [0.01, 0.1, 0.2],
    'min_child_weight': [1, 5, 10]
}

xgb_base = XGBClassifier(random_state=42, eval_metric='logloss')

grid_search = GridSearchCV(xgb_base, param_grid, scoring='roc_auc', cv=5, n_jobs=-1, verbose=1).fit(X_train, y_train)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV ROC-AUC: {grid_search.best_score_:.3f}')

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters: {'learning_rate': 0.1, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 50}
Best CV ROC-AUC: 0.812


In [26]:
xgb_tuned = XGBClassifier(**grid_search.best_params_, random_state=42, eval_metric='logloss').fit(X_train, y_train)

y_pred_tuned = xgb_tuned.predict(X_test)
y_pred_proba_tuned = xgb_tuned.predict_proba(X_test)[:, 1]

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_tuned))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_tuned))

print(f'\nROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_tuned):.3f}')

Confusion Matrix:
[[354 162]
 [111 369]]

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.69      0.72       516
           1       0.69      0.77      0.73       480

    accuracy                           0.73       996
   macro avg       0.73      0.73      0.73       996
weighted avg       0.73      0.73      0.73       996


ROC-AUC Score: 0.806


### Model Comparison (Updated)

| Metric | Logistic Regression | XGBoost (default) | XGBoost (tuned) |
|---|---|---|---|
| ROC-AUC | 0.791 | 0.773 | **0.806** |
| Accuracy | 0.70 | 0.68 | **0.73** |
| Recall (churned) | 0.68 | 0.68 | **0.77** |
| Precision (churned) | 0.70 | 0.67 | 0.69 |

Untuned XGBoost underperformed logistic regression, consistent with the small, mostly-linear feature set. After tuning (grid search, 5-fold CV, optimizing for ROC-AUC), XGBoost converged on a much shallower model (max_depth=2, n_estimators=50) than its defaults — and clearly outperformed logistic regression, most notably in recall (0.77 vs 0.68), meaning it catches substantially more true churners. **Tuned XGBoost is selected as the final model.**